# CLIK Workshop 2 - End-to-End ML di Snowflake (Enhanced)
## Use case: Probability of Default (PD) - Credit Default Prediction

**Enhancements over v1:**
- **Snowpark ML** preprocessing (distributed, no pandas bottleneck)
- **Feature Store** (entity, feature view, dataset generation)
- **ML Monitoring / MLOps** (model monitor, drift detection)

**Alur:**
1. Setup & Session
2. Feature Store: Entity + Feature View
3. Dataset generation dari Feature Store
4. Preprocessing dengan Snowpark ML
5. Model training: XGBoost & LightGBM (Snowpark ML distributed)
6. Logistic Regression (stepwise - pandas, karena stepwise belum ada di Snowpark ML)
7. Evaluasi & perbandingan
8. Register ke Model Registry
9. ML Monitoring (Model Monitor)
10. Test inference

Ref:
- https://www.snowflake.com/en/developers/guides/end-to-end-ml-workflow/
- https://www.snowflake.com/en/developers/guides/intro-to-online-feature-store-in-snowflake/

## 1. Setup & Session

### Cek environment (paket ML)

**Apa yang dilakukan:** Mengecek versi library `snowflake-ml-python` yang sudah terpasang, dan meng-upgrade **hanya bila** versinya terlalu lama.

**Kenapa ini penting:** Semua kemampuan ML native Snowflake (Feature Store, training, Model Registry, monitoring, serving) berasal dari library ini. Fitur *real-time serving* (Module 1b) membutuhkan versi ≥ 1.25.0.

**Keunggulan di Snowflake:** Notebook Container Runtime sudah membawa ratusan paket ML siap pakai — tidak perlu menyiapkan mesin/VM sendiri. Instalasi tambahan cukup `!pip install`, tanpa perlu akses internet eksternal.

In [ ]:
# Container Runtime (SPCS): snowflake-ml-python + core ML libs (snowpark, scikit-learn,
# xgboost, lightgbm, pandas, numpy, matplotlib) SUDAH preinstalled. Paket tambahan
# dipasang via !pip install (PyPI mirror internal Snowflake, tanpa external access) —
# BUKAN lewat dropdown Packages (itu utk warehouse-runtime/legacy).
# Real-time SPCS Model Serving (Module 1b) butuh snowflake-ml-python >= 1.25.0,
# jadi kita cek dulu dan hanya upgrade bila perlu.
from importlib.metadata import version
from packaging.version import Version

_cur = version("snowflake-ml-python")
print("snowflake-ml-python terpasang:", _cur)
if Version(_cur) < Version("1.25.0"):
    print("Versi < 1.25.0 -> meng-upgrade ...")
    !pip install -q --upgrade snowflake-ml-python
    print("Upgrade selesai. RESTART session/kernel (menu atas) lalu jalankan ulang notebook dari awal.")
else:
    print("Versi sudah memenuhi (>= 1.25.0). Lanjut ke cell berikutnya.")

### Menyambung ke Snowflake

**Apa yang dilakukan:** Meng-import library dan membuka *session* Snowpark, lalu memilih database, schema, dan warehouse yang akan dipakai.

**Kenapa ini penting:** `session` adalah "jembatan" antara kode Python dan data di Snowflake. Semua perintah berikutnya berjalan melalui session ini.

**Keunggulan di Snowflake:** Di dalam Notebook, session dibuat otomatis lewat `get_active_session()` — tidak perlu menyimpan username/password di kode. Data tidak diunduh ke laptop; komputasi terjadi di dalam Snowflake.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, avg, when, lit

session = get_active_session()
session.use_database('CLIK_WORKSHOP2')
session.use_schema('PUBLIC')
session.use_warehouse('GEN2_SMALL')
print('Connected:', session.get_current_database(), session.get_current_schema())

DB = 'CLIK_WORKSHOP2'
SCHEMA = 'PUBLIC'
WH = 'GEN2_SMALL'
TARGET = 'DEFAULT_FLAG'

## 2. Feature Store: Entity & Feature View
Menggunakan `snowflake.ml.feature_store` untuk:
- Mendaftarkan **Entity** (subject_id sebagai primary key)
- Membuat **Feature View** dari engineered features
- Track lineage & metadata fitur secara terpusat

### Menginisialisasi Feature Store

**Apa yang dilakukan:** Membuka (atau membuat) **Feature Store** di database/schema kita.

**Kenapa Feature Store penting dalam siklus ML:** Fitur (variabel input model) sering dibuat ulang oleh orang/tim berbeda dengan definisi yang sedikit beda → hasil model jadi tidak konsisten dan sulit diaudit. Feature Store menjadi **satu sumber kebenaran**: fitur didefinisikan sekali, diberi versi, dan dipakai ulang lintas model/tim.

**Keunggulan di Snowflake:** Feature Store sudah **native** di dalam Snowflake — tidak perlu tool atau infrastruktur terpisah. Ia otomatis mengurus penyimpanan, penyegaran (refresh), versi, dan lineage fitur, langsung di atas data yang sudah ter-governance.

In [ ]:
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode

fs = FeatureStore(
    session=session,
    database=DB,
    name=SCHEMA,
    default_warehouse=WH,
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)
print('Feature Store initialized')

### Mendaftarkan Entity

**Apa yang dilakukan:** Mendefinisikan **Entity** `CREDIT_SUBJECT` dengan *join key* `SUBJECT_ID` (satu baris = satu debitur).

**Kenapa ini penting:** Entity memberi tahu Feature Store "fitur ini melekat pada objek apa" dan bagaimana cara menggabungkannya (join) dengan data lain. Inilah yang membuat fitur bisa dipakai ulang secara aman & konsisten.

**Keunggulan di Snowflake:** Cukup dideklarasikan sekali; Snowflake menyimpan definisi & join key-nya, sehingga penggabungan fitur ke label saat training dilakukan otomatis.

In [ ]:
# Register Entity - SUBJECT_ID sebagai join key
try:
    subject_entity = fs.get_entity('CREDIT_SUBJECT')
    print('Retrieved existing entity')
except:
    subject_entity = Entity(
        name='CREDIT_SUBJECT',
        join_keys=['SUBJECT_ID'],
        desc='Credit bureau subject - satu baris per debitur'
    )
    fs.register_entity(subject_entity)
    print('Registered new entity: CREDIT_SUBJECT')

### Menyiapkan kumpulan fitur

**Apa yang dilakukan:** Memilih 34 fitur inti dari tabel `SUBJECT_FEATURES` dan membuat 1 fitur turunan `DTI_RATIO` (rasio cicilan terhadap pendapatan).

**Kenapa ini penting:** Kualitas model sangat ditentukan oleh kualitas fitur ("*garbage in, garbage out*"). Di sini kita memilih sinyal-sinyal risiko kredit yang relevan (tunggakan, utilisasi kredit, jumlah inquiry, dll).

**Keunggulan di Snowflake:** Transformasi (memilih kolom, menghitung rasio) berjalan sebagai **Snowpark DataFrame** — dieksekusi di dalam Snowflake tanpa menarik jutaan baris ke memori laptop.

In [ ]:
# Buat Feature View dari SUBJECT_FEATURES
# Pilih fitur yang akan dikelola di Feature Store
snow_df = session.table('SUBJECT_FEATURES')

CORE_FEATURES = [
    'SUBJECT_ID', 'AGE', 'MONTHLY_INCOME', 'NUM_ACTIVE_LOANS',
    'NUM_CREDIT_CARDS', 'NUM_FINTECH_LOANS', 'NUM_BNPL_ACCOUNTS',
    'NUM_LENDERS', 'NUM_INQUIRIES_3M', 'NUM_INQUIRIES_12M',
    'MAX_DPD_12M', 'MAX_DPD_24M', 'NUM_LATE_PMT_12M',
    'OLDEST_ACCT_AGE_MONTHS', 'AVG_ACCT_AGE_MONTHS',
    'TOTAL_OUTSTANDING', 'TOTAL_CREDIT_LIMIT', 'CREDIT_UTILIZATION',
    'MONTHLY_INSTALLMENT', 'KOL_STATUS', 'DEPENDENTS',
    'GENDER', 'EMPLOYMENT_TYPE', 'EDUCATION', 'REGION_CODE',
    'CC_OUTSTANDING', 'KTA_OUTSTANDING', 'KPR_OUTSTANDING',
    'CC_UTIL', 'KTA_UTIL', 'PAYMENT_RATIO_3M', 'PAYMENT_RATIO_12M',
    'ONTIME_RATIO_12M', 'REVOLVING_RATIO', 'DTI_RATIO'
]

# DTI_RATIO belum ada di tabel, kita buat sebagai derived feature
feature_df = snow_df.select(CORE_FEATURES[:-1]).with_column(
    'DTI_RATIO',
    col('MONTHLY_INSTALLMENT') / when(col('MONTHLY_INCOME') == 0, lit(1)).otherwise(col('MONTHLY_INCOME'))
)

print('Feature DataFrame columns:', len(feature_df.columns))
feature_df.show(3)

### Mendaftarkan Feature View

**Apa yang dilakukan:** Membungkus kumpulan fitur menjadi **Feature View** bernama `CREDIT_BUREAU_FEATURES` (versi 1), lengkap dengan deskripsi tiap fitur dan jadwal refresh harian.

**Kenapa ini penting:** Feature View adalah definisi fitur yang *terversi & terdokumentasi*. Model lain cukup memanggil view ini alih-alih menghitung ulang fitur → hemat waktu, konsisten, dan bisa diaudit.

**Keunggulan di Snowflake:** `refresh_freq='1 day'` membuat fitur diperbarui **otomatis** oleh Snowflake. Deskripsi & lineage tersimpan, sehingga mudah dilacak siapa/model apa yang memakai fitur tertentu.

In [ ]:
# Register Feature View
credit_fv = FeatureView(
    name='CREDIT_BUREAU_FEATURES',
    entities=[subject_entity],
    feature_df=feature_df,
    refresh_freq='1 day',
    desc='Core credit bureau features for PD modeling (35 features)'
)

# Attach feature descriptions
credit_fv = credit_fv.attach_feature_desc({
    'AGE': 'Usia debitur',
    'MONTHLY_INCOME': 'Pendapatan bulanan (IDR)',
    'CREDIT_UTILIZATION': 'Rasio penggunaan kredit',
    'MAX_DPD_12M': 'Hari keterlambatan maksimum 12 bulan terakhir',
    'KOL_STATUS': 'Kolektibilitas OJK (1=lancar, 5=macet)',
    'DTI_RATIO': 'Debt-to-income ratio',
    'NUM_INQUIRIES_12M': 'Jumlah inquiry biro 12 bulan terakhir',
})

registered_fv = fs.register_feature_view(credit_fv, version='1', overwrite=True)
print('Feature View registered:', registered_fv.name, registered_fv.version)

## 3. Generate Training Dataset dari Feature Store
Gunakan `fs.generate_dataset()` — menggabungkan spine (label) dengan feature view secara otomatis.

### Membuat dataset training dari Feature Store

**Apa yang dilakukan:** Membuat "spine" (daftar `SUBJECT_ID` + label `DEFAULT_FLAG`), lalu `generate_dataset()` **otomatis menempelkan fitur** dari Feature View ke spine tersebut.

**Kenapa ini penting:** Ini memisahkan *label* (apa yang diprediksi) dari *fitur* (input). Feature Store menjamin fitur yang ditarik sesuai definisi resmi — mencegah *data leakage* dan ketidakcocokan antara data training dan data produksi.

**Keunggulan di Snowflake:** Penggabungan fitur↔label dikerjakan otomatis & efisien di dalam Snowflake. Dataset hasilnya tercatat (punya nama & versi), sehingga eksperimen dapat direproduksi.

In [ ]:
# Spine = ID + label (tanpa fitur, fitur ditarik dari Feature Store)
spine_df = snow_df.select('SUBJECT_ID', 'DEFAULT_FLAG').sample(n=300000)
print('Spine rows:', spine_df.count())

# Generate dataset - Feature Store auto-join fitur ke spine
ds = fs.generate_dataset(
    name='CLIK_PD_TRAINING_SET',
    spine_df=spine_df,
    features=[registered_fv],
    spine_label_cols=['DEFAULT_FLAG'],
    desc='PD training dataset from Feature Store'
)

# Read sebagai Snowpark DataFrame
ds_sp = ds.read.to_snowpark_dataframe()
print('Training dataset shape:', ds_sp.count(), 'rows x', len(ds_sp.columns), 'cols')
ds_sp.show(3)

## 4. Preprocessing dengan Snowpark ML
Ganti `sklearn.preprocessing` → `snowflake.ml.modeling.preprocessing`.
Process berjalan **distributed di Snowflake** (bukan di client/pandas).

### Memisahkan kolom kategorikal vs numerik

**Apa yang dilakukan:** Secara otomatis mendeteksi kolom teks (kategorikal, mis. GENDER) dan kolom angka (numerik) dari skema dataset.

**Kenapa ini penting:** Kedua jenis kolom butuh perlakuan berbeda — yang teks perlu diubah jadi angka dulu. Memisahkannya membuat langkah preprocessing berikutnya tepat sasaran.

**Keunggulan di Snowflake:** Tipe data sudah melekat pada Snowpark DataFrame, jadi deteksi tipe akurat tanpa menebak-nebak.

In [ ]:
import snowflake.ml.modeling.preprocessing as snowml
from snowflake.snowpark.types import StringType

# Identifikasi kolom kategorikal
CAT_COLS = [c.name for c in ds_sp.schema if c.datatype == StringType() and c.name != 'SUBJECT_ID']
NUM_COLS = [c for c in ds_sp.columns if c not in CAT_COLS + ['SUBJECT_ID', 'DEFAULT_FLAG']]

print(f'Categorical: {len(CAT_COLS)} -> {CAT_COLS}')
print(f'Numeric: {len(NUM_COLS)}')

### One-Hot Encoding (terdistribusi)

**Apa yang dilakukan:** Mengubah kolom kategorikal (teks) menjadi kolom angka 0/1 memakai `OneHotEncoder` dari **Snowpark ML**.

**Kenapa ini penting:** Algoritma ML hanya bisa berhitung dengan angka. Encoding adalah syarat agar variabel seperti GENDER atau EMPLOYMENT_TYPE bisa dipakai model.

**Keunggulan di Snowflake:** Berbeda dengan `sklearn` yang berjalan di satu mesin, encoder Snowpark ML dieksekusi **terdistribusi di warehouse** — mampu memproses data besar tanpa kehabisan memori di laptop.

> **Catatan teknis:** `OneHotEncoder` menghasilkan nama kolom dengan karakter khusus (tanda kutip/spasi/koma). XGBoost menerimanya, tetapi **LightGBM menolak** (`special JSON characters in feature name`). Karena itu di sel berikut nama kolom kita **bersihkan** dulu sebelum training.

In [ ]:
# OneHotEncoder (Snowpark ML) - berjalan distributed di warehouse
ohe = snowml.OneHotEncoder(
    input_cols=CAT_COLS,
    output_cols=CAT_COLS,
    drop_input_cols=True
)
ds_encoded = ohe.fit(ds_sp).transform(ds_sp)
print('After OHE:', len(ds_encoded.columns), 'columns')

# --- PENTING: bersihkan nama kolom hasil OHE ---
# OneHotEncoder menghasilkan nama kolom dengan karakter khusus (tanda kutip, spasi,
# koma, titik dua, kurung, dll). XGBoost bisa mentolerir, TAPI LightGBM menolak
# ("Do not support special JSON characters in feature name"). Kita rapikan dulu.
import re
def _clean_name(name):
    n = name.strip('"')                       # buang tanda kutip pembungkus
    n = re.sub(r'[^0-9A-Za-z_]', '_', n)      # ganti karakter non-alfanumerik jadi '_'
    n = re.sub(r'_+', '_', n).strip('_')      # rapikan '_' berulang
    return n.upper()

new_names, seen = [], {}
for c in ds_encoded.columns:
    nc = _clean_name(c)
    if nc in seen:                            # jaga keunikan nama
        seen[nc] += 1
        nc = f"{nc}_{seen[nc]}"
    else:
        seen[nc] = 0
    new_names.append(nc)

# to_df me-rename SEMUA kolom sekaligus secara posisional (paling robust)
ds_encoded = ds_encoded.to_df(new_names)
print('Nama kolom sudah dibersihkan untuk kompatibilitas LightGBM')

# StandardScaler (Snowpark ML) - opsional untuk tree models, wajib untuk LR
# Untuk XGB/LGBM kita skip scaling, untuk LR stepwise tetap pakai
# (karena stepwise LR butuh normalisasi)

### Membagi data Train / Test

**Apa yang dilakukan:** Memisahkan data menjadi 75% untuk melatih model dan 25% untuk menguji, memakai `random_split()` native Snowpark.

**Kenapa ini penting:** Menguji model pada data yang **tidak** dilihat saat training adalah cara jujur mengukur performa dan menghindari *overfitting* (model yang hanya hafal, bukan belajar).

**Keunggulan di Snowflake:** Pembagian dilakukan langsung di Snowflake tanpa `to_pandas()`, sehingga tetap cepat dan sanggup menangani data berskala besar.

In [ ]:
# Train/Test split menggunakan random_split (native Snowpark, no pandas!)
train_sp, test_sp = ds_encoded.random_split(weights=[0.75, 0.25], seed=42)
train_sp = train_sp.fillna(0)
test_sp = test_sp.fillna(0)

print('Train:', train_sp.count(), '| Test:', test_sp.count())

# Kolom fitur (semua kecuali ID dan target)
FEATURE_COLS = [c for c in train_sp.columns if c not in ['SUBJECT_ID', 'DEFAULT_FLAG']]
print('Feature cols for training:', len(FEATURE_COLS))

## 5. Model Training: XGBoost (Snowpark ML Distributed)
Gunakan `snowflake.ml.modeling.xgboost.XGBClassifier` — training berjalan **di warehouse** (distributed), bukan di client.

### Melatih model XGBoost (terdistribusi)

**Apa yang dilakukan:** Melatih `XGBClassifier` (Snowpark ML) untuk memprediksi peluang gagal bayar. Parameter `scale_pos_weight=10` memberi bobot lebih pada kelas minoritas (nasabah gagal bayar yang hanya ~8%).

**Kenapa ini penting:** XGBoost adalah salah satu algoritma terkuat untuk data tabular seperti skoring kredit — sering menang di kompetisi & dipakai luas di industri keuangan.

**Keunggulan di Snowflake:** Training berjalan **di dalam warehouse Snowflake**, bukan di laptop. Data tidak keluar dari Snowflake (aman & ter-governance), dan kapasitas bisa dinaikkan cukup dengan memperbesar ukuran warehouse.

In [ ]:
from snowflake.ml.modeling.xgboost import XGBClassifier
from snowflake.ml.modeling.metrics import roc_auc_score as snow_auc

xgb_model = XGBClassifier(
    input_cols=FEATURE_COLS,
    label_cols=[TARGET],
    output_cols=['XGB_PREDICTION'],
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=10,  # ~10:1 class imbalance
    random_state=42
)

print('Training XGBoost (distributed di warehouse)...')
xgb_model.fit(train_sp)
print('XGBoost trained!')

### Mengevaluasi XGBoost

**Apa yang dilakukan:** Menjalankan prediksi pada data test lalu menghitung **AUC** & **Gini** — ukuran seberapa baik model memisahkan nasabah yang lancar vs berisiko gagal bayar.

**Kenapa ini penting:** Inilah angka yang dipakai bisnis untuk memutuskan apakah model layak dipakai. Menguji di data yang belum pernah dilihat model = penilaian yang jujur.

**Keunggulan di Snowflake:** Prediksi dihitung di dalam Snowflake; kita hanya menarik kolom kecil (label + prediksi) ke pandas untuk menghitung metrik — efisien walau data besar.

In [ ]:
# Evaluate XGBoost
xgb_preds = xgb_model.predict(test_sp)
xgb_preds_pd = xgb_preds.select('DEFAULT_FLAG', 'XGB_PREDICTION').to_pandas()

from sklearn.metrics import roc_auc_score, classification_report
xgb_auc = roc_auc_score(xgb_preds_pd['DEFAULT_FLAG'], xgb_preds_pd['XGB_PREDICTION'])
print(f'XGBoost - AUC: {xgb_auc:.4f} | Gini: {2*xgb_auc-1:.4f}')
print(classification_report(xgb_preds_pd['DEFAULT_FLAG'], xgb_preds_pd['XGB_PREDICTION'], digits=3))

## 6. Model Training: LightGBM (Snowpark ML Distributed)

### Melatih model LightGBM (terdistribusi)

**Apa yang dilakukan:** Melatih alternatif model, `LGBMClassifier`, dengan `class_weight='balanced'` untuk menangani data yang tidak seimbang (nasabah gagal bayar hanya ~8%).

**Kenapa ini penting:** Mencoba lebih dari satu algoritma lalu membandingkannya adalah praktik standar untuk memilih model terbaik — bukan bergantung pada satu algoritma saja.

**Keunggulan di Snowflake:** Sama seperti XGBoost, training berjalan **terdistribusi di warehouse** memakai data & pipeline yang sama, sehingga perbandingan konsisten dan skalabel.

In [ ]:
from snowflake.ml.modeling.lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(
    input_cols=FEATURE_COLS,
    label_cols=[TARGET],
    output_cols=['LGBM_PREDICTION'],
    n_estimators=400,
    num_leaves=48,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight='balanced',
    random_state=42,
    verbose=-1
)

print('Training LightGBM (distributed di warehouse)...')
lgbm_model.fit(train_sp)
print('LightGBM trained!')

### Mengevaluasi LightGBM

**Apa yang dilakukan:** Menghitung AUC & Gini LightGBM pada data test, supaya bisa diadu dengan XGBoost dan Logistic Regression di langkah perbandingan.

**Kenapa ini penting:** Perbandingan yang adil mengharuskan setiap model diukur dengan metrik yang sama pada data test yang sama.

In [ ]:
lgbm_preds = lgbm_model.predict(test_sp)
lgbm_preds_pd = lgbm_preds.select('DEFAULT_FLAG', 'LGBM_PREDICTION').to_pandas()
lgbm_auc = roc_auc_score(lgbm_preds_pd['DEFAULT_FLAG'], lgbm_preds_pd['LGBM_PREDICTION'])
print(f'LightGBM - AUC: {lgbm_auc:.4f} | Gini: {2*lgbm_auc-1:.4f}')

## 7. Model Training: Logistic Regression (Stepwise)
Stepwise forward selection belum tersedia di Snowpark ML.
Kita gunakan pandas sampling + sklearn LR untuk step ini (realistic approach).

### Memilih fitur untuk Logistic Regression (langkah 1: ranking)

**Apa yang dilakukan:** Menarik sampel data ke pandas, lalu memeringkat fitur numerik berdasarkan kekuatan prediktifnya masing-masing (*univariate AUC*).

**Kenapa ini penting:** Sebelum membangun model sederhana yang mudah dijelaskan, kita cari dulu fitur mana yang paling "berbicara" tentang risiko gagal bayar.

**Catatan Snowflake:** *Stepwise selection* belum tersedia di Snowpark ML, jadi langkah ini sengaja memakai pandas + sklearn pada **sampel** (bukan seluruh 1 juta baris) — pendekatan realistis untuk teknik yang belum terdistribusi. Model utama (XGBoost/LightGBM) tetap dilatih terdistribusi di Snowflake.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler as SkScaler

# Tarik subset ke pandas untuk stepwise
LR_SAMPLE = 100_000
train_lr_pd = train_sp.sample(n=LR_SAMPLE).to_pandas()
test_lr_pd = test_sp.to_pandas()

NUM_ONLY = [c for c in FEATURE_COLS if not any(cat in c for cat in CAT_COLS)]

# Univariate AUC ranking
uni = {}
for c in NUM_ONLY[:50]:
    try:
        uni[c] = abs(roc_auc_score(train_lr_pd[TARGET], train_lr_pd[c].fillna(0)) - 0.5)
    except:
        uni[c] = 0
ranked = sorted(uni, key=uni.get, reverse=True)[:30]
print('Top 10 features:', ranked[:10])

### Pemilihan fitur *forward stepwise* (langkah 2)

**Apa yang dilakukan:** Menambahkan fitur satu per satu; sebuah fitur hanya dipertahankan jika ia menaikkan AUC pada data validasi.

**Kenapa ini penting:** Menghasilkan model yang **ramping** — hanya berisi fitur yang benar-benar berkontribusi. Model ramping lebih stabil, lebih mudah dipantau, dan lebih mudah dijelaskan.

In [ ]:
from sklearn.model_selection import train_test_split as tts

Xa, Xv, ya, yv = tts(train_lr_pd[ranked], train_lr_pd[TARGET], test_size=0.3, random_state=7, stratify=train_lr_pd[TARGET])
scaler = SkScaler()
Xa_s = scaler.fit_transform(Xa.fillna(0))
Xv_s = scaler.transform(Xv.fillna(0))

# Forward stepwise
selected_idx, best_auc = [], 0.5
for _ in range(20):
    trial_best, trial_col = best_auc, None
    for i, c in enumerate(ranked):
        if i in selected_idx: continue
        cols = selected_idx + [i]
        lr = LogisticRegression(max_iter=200, C=1.0)
        lr.fit(Xa_s[:, cols], ya)
        auc = roc_auc_score(yv, lr.predict_proba(Xv_s[:, cols])[:, 1])
        if auc > trial_best + 1e-4:
            trial_best, trial_col = auc, i
    if trial_col is None: break
    selected_idx.append(trial_col); best_auc = trial_best
    print(f'+ {ranked[trial_col]:30s} -> AUC {best_auc:.4f} ({len(selected_idx)} feats)')

selected_features = [ranked[i] for i in selected_idx]
print('\nSelected features:', selected_features)

### Melatih Logistic Regression final

**Apa yang dilakukan:** Melatih model Logistic Regression memakai fitur-fitur terpilih, lalu mengukur AUC/Gini pada data test.

**Kenapa ini penting:** Logistic Regression menjadi *baseline* yang **transparan dan mudah dijelaskan** — pembanding penting terhadap model yang lebih kompleks (XGBoost/LightGBM). Di industri kredit, kemampuan menjelaskan keputusan sering diwajibkan regulator.

In [ ]:
# Final LR on full train
X_train_lr = scaler.fit_transform(train_lr_pd[selected_features].fillna(0))
X_test_lr = scaler.transform(test_lr_pd[selected_features].fillna(0))
lr_final = LogisticRegression(max_iter=500, C=1.0)
lr_final.fit(X_train_lr, train_lr_pd[TARGET])
lr_pred = lr_final.predict_proba(X_test_lr)[:, 1]
lr_auc = roc_auc_score(test_lr_pd[TARGET], lr_pred)
print(f'Logistic Regression (stepwise) - AUC: {lr_auc:.4f} | Gini: {2*lr_auc-1:.4f}')

## 8. Model Comparison & ROC Curves

### Membandingkan model & kurva ROC

**Apa yang dilakukan:** Menyusun tabel AUC/Gini untuk ketiga model dan menggambar **kurva ROC** sebagai perbandingan visual.

**Kenapa ini penting:** Memilih model bukan sekadar angka tertinggi — kita juga menimbang kompleksitas vs kemudahan menjelaskan ke regulator. Kurva ROC memperlihatkan trade-off antara menangkap nasabah berisiko vs salah menolak nasabah baik.

**Istilah singkat:** **AUC** = 0.5 berarti setara tebak-tebakan, mendekati 1.0 berarti sangat baik memisahkan nasabah baik vs berisiko. **Gini** = 2×AUC−1.

In [ ]:
from sklearn.metrics import roc_curve

results = pd.DataFrame({
    'Model': ['LogReg (stepwise)', 'XGBoost (Snowpark ML)', 'LightGBM (Snowpark ML)'],
    'AUC':  [lr_auc, xgb_auc, lgbm_auc],
    'Gini': [2*lr_auc-1, 2*xgb_auc-1, 2*lgbm_auc-1],
}).sort_values('AUC', ascending=False).reset_index(drop=True)
display(results)

plt.figure(figsize=(7,5))
# LR
fpr, tpr, _ = roc_curve(test_lr_pd[TARGET], lr_pred)
plt.plot(fpr, tpr, label=f'LogReg (AUC={lr_auc:.3f})')
# XGB
fpr, tpr, _ = roc_curve(xgb_preds_pd['DEFAULT_FLAG'], xgb_preds_pd['XGB_PREDICTION'])
plt.plot(fpr, tpr, label=f'XGBoost (AUC={xgb_auc:.3f})')
# LGBM
fpr, tpr, _ = roc_curve(lgbm_preds_pd['DEFAULT_FLAG'], lgbm_preds_pd['LGBM_PREDICTION'])
plt.plot(fpr, tpr, label=f'LightGBM (AUC={lgbm_auc:.3f})')
plt.plot([0,1],[0,1],'k--',alpha=0.4)
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC Curves - PD Models (Snowpark ML)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### Visualisasi interaktif dengan Altair

**Apa yang dilakukan:** Menggambar ulang **kurva ROC** dan **bar chart AUC** memakai Altair — grafik interaktif (bisa di-*hover* untuk melihat angka & di-zoom).

**Kenapa ini penting:** Grafik jauh lebih mudah dijelaskan ke audiens non-teknis dibanding tabel angka. Model dengan kurva paling mendekati sudut **kiri-atas** dan **bar AUC tertinggi** adalah yang terbaik.

**Cara membacanya:** Garis putus-putus diagonal = tebak-tebakan (AUC 0.5). Makin jauh kurva sebuah model berada **di atas** garis diagonal itu, makin baik ia memisahkan nasabah lancar vs berisiko gagal bayar.

In [ ]:
# Grafik evaluasi. Coba Altair (interaktif); bila TIDAK tersedia di runtime,
# otomatis fallback ke matplotlib (selalu terpasang) TANPA pip install.
# Catatan: Container Runtime bisa tanpa akses internet, jadi jangan andalkan pip.
from sklearn.metrics import roc_curve, roc_auc_score

_candidates = [
    ('LogReg (stepwise)', test_lr_pd[TARGET],            lr_pred),
    ('XGBoost',           xgb_preds_pd['DEFAULT_FLAG'],  xgb_preds_pd['XGB_PREDICTION']),
    ('LightGBM',          lgbm_preds_pd['DEFAULT_FLAG'], lgbm_preds_pd['LGBM_PREDICTION']),
]

try:
    import altair as alt
    roc_frames = []
    for name, y_true, y_score in _candidates:
        fpr, tpr, _ = roc_curve(y_true, y_score)
        roc_frames.append(pd.DataFrame({'FPR': fpr, 'TPR': tpr, 'Model': name}))
    roc_long = pd.concat(roc_frames, ignore_index=True)

    roc_lines = alt.Chart(roc_long).mark_line(size=2.5).encode(
        x=alt.X('FPR:Q', title='False Positive Rate'),
        y=alt.Y('TPR:Q', title='True Positive Rate'),
        color=alt.Color('Model:N', title='Model'),
        tooltip=['Model:N', alt.Tooltip('FPR:Q', format='.3f'), alt.Tooltip('TPR:Q', format='.3f')],
    )
    diag = alt.Chart(pd.DataFrame({'x': [0, 1], 'y': [0, 1]})).mark_line(
        strokeDash=[4, 4], color='gray').encode(x='x:Q', y='y:Q')
    roc_chart = (roc_lines + diag).properties(
        width=440, height=360, title='Kurva ROC — makin dekat sudut kiri-atas makin baik').interactive()

    bars = alt.Chart(results).mark_bar().encode(
        x=alt.X('AUC:Q', scale=alt.Scale(domain=[0.5, 1.0]), title='AUC'),
        y=alt.Y('Model:N', sort='-x', title=None),
        color=alt.Color('Model:N', legend=None),
        tooltip=['Model:N', alt.Tooltip('AUC:Q', format='.4f'), alt.Tooltip('Gini:Q', format='.4f')],
    )
    labels = bars.mark_text(align='left', dx=3).encode(text=alt.Text('AUC:Q', format='.3f'))
    auc_chart = (bars + labels).properties(width=440, height=180, title='Perbandingan AUC antar model')

    display(alt.vconcat(roc_chart, auc_chart).resolve_scale(color='independent'))

except ModuleNotFoundError:
    print('Altair tidak tersedia di runtime ini -> memakai matplotlib.')
    import matplotlib.pyplot as plt
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    # ROC
    for name, y_true, y_score in _candidates:
        fpr, tpr, _ = roc_curve(y_true, y_score)
        ax1.plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(y_true, y_score):.3f})')
    ax1.plot([0, 1], [0, 1], 'k--', alpha=0.4)
    ax1.set_xlabel('False Positive Rate'); ax1.set_ylabel('True Positive Rate')
    ax1.set_title('Kurva ROC — makin dekat sudut kiri-atas makin baik')
    ax1.legend(); ax1.grid(alpha=0.3)

    # Bar AUC
    r = results.sort_values('AUC')
    ax2.barh(r['Model'], r['AUC'], color=['#8ecae6', '#219ebc', '#023047'][:len(r)])
    ax2.set_xlim(0.5, 1.0); ax2.set_xlabel('AUC')
    ax2.set_title('Perbandingan AUC antar model')
    for i, v in enumerate(r['AUC']):
        ax2.text(v + 0.003, i, f'{v:.3f}', va='center')
    plt.tight_layout(); plt.show()

## 9. Register Best Model ke Model Registry
Gunakan `enable_monitoring=True` untuk menyiapkan ML Observability.

### Menyimpan model ke Model Registry

**Apa yang dilakukan:** Mendaftarkan model terbaik sebagai `CLIK_PD_MODEL` versi `V2_SNOWPARK_ML` — lengkap dengan contoh input, metrik (AUC/Gini), dan komentar — lalu menjadikannya versi *default*.

**Kenapa Model Registry penting dalam siklus ML:** Tanpa registry, model tersimpan sebagai file (pickle) yang mudah hilang atau tertukar. Registry memberi **versi, metrik, lineage, dan tata-kelola** sehingga kita tahu persis model mana yang dipakai di produksi dan seberapa bagus performanya.

**Keunggulan di Snowflake:** Registry native — model langsung bisa dipakai untuk inference (SQL batch maupun REST real-time) tanpa memindahkan artefak. Opsi `enable_monitoring=True` sekaligus menyiapkan pemantauan otomatis.

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.model import type_hints as th

best_name = results.iloc[0]['Model']
best_model = xgb_model if 'XGBoost' in best_name else lgbm_model

reg = Registry(
    session=session, database_name=DB, schema_name=SCHEMA,
    options={'enable_monitoring': True}
)

MODEL_NAME, VERSION = 'CLIK_PD_MODEL', 'V2_SNOWPARK_ML'

# Idempotent (aman untuk re-run): jika versi sudah ada, hapus dulu supaya bisa
# di-log ulang dengan target_platforms yang benar. Monitor yang mereferensikan
# versi harus di-drop lebih dulu agar delete_version tidak gagal.
try:
    session.sql('DROP MODEL MONITOR IF EXISTS CLIK_PD_MODEL_MONITOR').collect()
except Exception as e:
    print('skip drop monitor:', e)
try:
    reg.get_model(MODEL_NAME).delete_version(VERSION)
    print('Deleted existing version', VERSION)
except Exception:
    pass  # versi belum ada -> lanjut log baru

# Log model terbaik.
# - task=... WAJIB untuk Model Monitor (PD = binary classification)
# - target_platforms=... WAJIB agar bisa inference di WAREHOUSE (default Container
#   Runtime hanya SPCS). Sertakan keduanya supaya batch (warehouse) & real-time (SPCS) jalan.
mv = reg.log_model(
    model=best_model,
    model_name=MODEL_NAME,
    version_name=VERSION,
    sample_input_data=train_sp.drop(['SUBJECT_ID', 'DEFAULT_FLAG']).limit(100),
    comment=f'PD model ({best_name}) trained with Snowpark ML. AUC={results.iloc[0]["AUC"]:.4f}',
    conda_dependencies=['xgboost', 'lightgbm', 'scikit-learn'],
    task=th.Task.TABULAR_BINARY_CLASSIFICATION,
    target_platforms=['WAREHOUSE', 'SNOWPARK_CONTAINER_SERVICES'],
)

# Set metrics
mv.set_metric(metric_name='Test_AUC', value=float(results.iloc[0]['AUC']))
mv.set_metric(metric_name='Test_Gini', value=float(results.iloc[0]['Gini']))
print('Registered:', mv.model_name, mv.version_name)

# Set as default version
reg.get_model(MODEL_NAME).default = VERSION

## 10. ML Monitoring / MLOps
Snowflake Model Monitor tracks:
- **Prediction drift** (distribusi prediksi bergeser dari baseline)
- **Feature drift** (distribusi input berubah)
- **Performance metrics** (accuracy/AUC jika ground truth tersedia)

Ref: end-to-end-ml-workflow guide — `CREATE MODEL MONITOR`

### Menyiapkan data untuk monitoring

**Apa yang dilakukan:** Menyimpan dua tabel — **baseline** (data training + prediksi) dan **source** (data "produksi" + prediksi) — masing-masing diberi kolom waktu `SCORED_AT`.

**Kenapa ini penting:** Monitoring bekerja dengan cara membandingkan kondisi "sekarang/produksi" terhadap "baseline". Kedua snapshot ini adalah bahan bakar agar drift bisa dihitung.

**Keunggulan di Snowflake:** Menulis hasil prediksi kembali ke tabel cukup satu baris `write.save_as_table()` — datanya tetap di dalam Snowflake, siap dipantau.

In [ ]:
# Simpan train & test data sebagai tabel untuk monitoring baseline & source
# Tambahkan kolom prediksi dan timestamp

# PENTING: di Workspace notebook, database aktif bisa jatuh ke "personal database"
# (tempat CREATE TABLE diblokir). Pastikan konteks ke DB workshop + fully-qualify nama tabel.
session.use_database(DB)
session.use_schema(SCHEMA)
BASELINE_TBL = f'{DB}.{SCHEMA}.PD_MONITOR_BASELINE'
SOURCE_TBL   = f'{DB}.{SCHEMA}.PD_MONITOR_SOURCE'

# 1) Baseline = training data + predictions
train_with_preds = xgb_model.predict(train_sp).with_column('SCORED_AT', lit('2026-07-01 00:00:00').cast('TIMESTAMP_NTZ'))
train_with_preds.write.save_as_table(BASELINE_TBL, mode='overwrite')
print('Baseline table saved:', session.table(BASELINE_TBL).count(), 'rows')

# 2) Source = test/production data + predictions (simulates production scoring)
test_with_preds = xgb_model.predict(test_sp).with_column('SCORED_AT', lit('2026-07-15 00:00:00').cast('TIMESTAMP_NTZ'))
test_with_preds.write.save_as_table(SOURCE_TBL, mode='overwrite')
print('Source table saved:', session.table(SOURCE_TBL).count(), 'rows')

### Membuat Model Monitor

**Apa yang dilakukan:** Membuat objek `MODEL MONITOR` yang secara berkala membandingkan prediksi & input "produksi" (source) terhadap "baseline".

**Kenapa monitoring penting dalam siklus ML:** Sebuah model tidak selamanya akurat — perilaku nasabah dan kondisi ekonomi berubah. Tanpa monitoring, penurunan performa baru ketahuan setelah bisnis dirugikan.

**Keunggulan di Snowflake:** Monitoring adalah **objek native** (`CREATE MODEL MONITOR`) dengan refresh otomatis. Tidak perlu membangun pipeline pemantauan terpisah; hasilnya langsung terlihat di Snowsight.

In [ ]:
# Pastikan task model sudah diset (WAJIB untuk Model Monitor).
# Aman dijalankan walau task sudah diset (mis. model dilog sebelum perbaikan).
session.sql(f"ALTER MODEL {DB}.{SCHEMA}.CLIK_PD_MODEL MODIFY VERSION V2_SNOWPARK_ML "
            "SET TASK = TABULAR_BINARY_CLASSIFICATION").collect()

# Buat Model Monitor via SQL
session.sql('''
CREATE OR REPLACE MODEL MONITOR CLIK_PD_MODEL_MONITOR
WITH
    MODEL = CLIK_PD_MODEL
    VERSION = V2_SNOWPARK_ML
    FUNCTION = predict
    SOURCE = CLIK_WORKSHOP2.PUBLIC.PD_MONITOR_SOURCE
    BASELINE = CLIK_WORKSHOP2.PUBLIC.PD_MONITOR_BASELINE
    TIMESTAMP_COLUMN = SCORED_AT
    PREDICTION_CLASS_COLUMNS = (XGB_PREDICTION)
    ACTUAL_CLASS_COLUMNS = (DEFAULT_FLAG)
    ID_COLUMNS = (SUBJECT_ID)
    WAREHOUSE = GEN2_SMALL
    REFRESH_INTERVAL = '1 hour'
    AGGREGATION_WINDOW = '1 day'
''').collect()
print('Model Monitor created: CLIK_PD_MODEL_MONITOR')

### Melihat metrik drift

**Apa yang dilakukan:** Query fungsi bawaan `MODEL_MONITOR_DRIFT_METRIC` untuk melihat pergeseran (*drift*) distribusi prediksi pada rentang waktu tertentu.

**Kenapa ini penting:** *Drift* adalah sinyal dini bahwa model mulai "meleset" dari kenyataan dan mungkin perlu dilatih ulang. Memantaunya rutin menjaga keputusan kredit tetap andal.

**Keunggulan di Snowflake:** Cukup satu query SQL — tidak perlu membangun sistem penghitung drift sendiri. Hasilnya juga tampil di Snowsight (**AI & ML → Models**).

In [ ]:
# Query drift metrics
drift_df = session.sql('''
SELECT * FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
    'CLIK_PD_MODEL_MONITOR',
    'DIFFERENCE_OF_MEANS',
    'XGB_PREDICTION',
    '1 DAY',
    DATEADD(DAY, -30, CURRENT_DATE()),
    CURRENT_DATE()
))
''')
drift_df.show()
print('\nDrift analysis complete. Monitor juga bisa dilihat di Snowsight > AI & ML > Models.')

## 11. Test Inference dari Registry

### Uji coba inference dari Model Registry

**Apa yang dilakukan:** Mengambil 10 baris contoh, lalu memanggil `mv.run(..., function_name='predict')` — menjalankan model langsung dari Registry untuk memprediksi peluang gagal bayar.

**Kenapa ini penting:** Ini membuktikan model yang tersimpan benar-benar bisa dipakai untuk data baru. Ini "jembatan" menuju deployment nyata (Module 1b: batch & real-time scoring).

**Keunggulan di Snowflake:** Kita memanggil model dari Registry tanpa memuat ulang file model. Model yang sama bisa dipanggil dari **Python maupun SQL** — sekali daftar, dipakai di mana saja.

In [ ]:
# Batch inference via model registry
test_sdf = session.table('SUBJECT_FEATURES').limit(10)
pred_sdf = mv.run(test_sdf, function_name='predict')
pred_sdf.select('SUBJECT_ID', '"output_feature_0"').show()

## Summary - Snowflake ML/MLOps Stack yang Digunakan

| Layer | sklearn (sebelumnya) | Snowpark ML (sekarang) |
|-------|---------------------|------------------------|
| Preprocessing | `sklearn.preprocessing.OneHotEncoder` | `snowflake.ml.modeling.preprocessing.OneHotEncoder` |
| Train/Test Split | `sklearn.model_selection.train_test_split` | `DataFrame.random_split()` |
| Training | `xgboost.XGBClassifier` (local) | `snowflake.ml.modeling.xgboost.XGBClassifier` (distributed) |
| Feature Management | Manual column selection | **Feature Store** (Entity, FeatureView, generate_dataset) |
| Model Storage | File-based / pickle | **Model Registry** (versioning, metrics, tags) |
| Monitoring | Manual / none | **Model Monitor** (drift detection, performance tracking) |
| Deployment | Custom SPCS | **Model Serving** (create_service REST API) |

Lanjut ke:
- **04a** — Batch scoring (warehouse)
- **04b** — Real-time inference (SPCS)